# Mini-Projeto: Modelagem e Solução de Problemas com Aprendizado por Reforço

**Alunos:** Deborah Ferreira 599070, Andrezza, João Lucas e Henrique Sanchez

Neste notebook, vocês irão desenvolver o projeto prático da disciplina. O objetivo é aplicar os algoritmos tabulares clássicos em um ambiente de espaço contínuo (`MountainCar-v0`), utilizando técnicas de discretização, conforme o enunciado.

Deixe, neste colab, somente o código necessário para treinar os agentes com a parametrização mais vantajosa e gerar os gráficos, além do relatório em texto, na **análise dos resultados** e no **diário de bordo**. O restante da experimentação pode ser feito em outros colabs e ambientes. O diário de bordo deve comentar essa jornada e pode referenciar esses outros artefatos.

## Tarefa A: Modelagem do Espaço de Estados (Discretização)

Implementem abaixo **duas formas ou resoluções de discretização diferentes**

In [ ]:
# ==============================================================================
# TAREFA A: FUNÇÕES DE DISCRETIZAÇÃO
# ==============================================================================

import gymnasium as gym
from gymnasium import spaces
import numpy as np

POS_MINIMA = -1.2
POS_MAXIMA = 0.6
VEL_MINIMA = -0.07
VEL_MAXIMA = 0.07

# Discretização por Grid Uniforme => classe para transformar o espaço contínuo de estados
# do Mountain Car em um grid com tamanho de linhas e de colunas passadas por parâmetros
# 1) Discretização grosseira: 10 posições diferentes e 10 velocidades possíveis
# 2) Discretização fina: 30 posições diferentes e 30 velocidades possíveis
# (passar como parâmetros)

class StateDiscretizer:
    def __init__(self, n_posicoes, n_velocidades):
        # calcula o tamanho do bloco ocupado por cada eixo dado o n

        self.tamanho_degrau_pos = (POS_MAXIMA - POS_MINIMA) / n_posicoes
        self.tamanho_degrau_vel = (VEL_MAXIMA - VEL_MINIMA) / n_velocidades

        self.n_posicoes = n_posicoes
        self.n_velocidades = n_velocidades

    def discretize(self, state):
        pos, vel = state
        # calcula a posição e a velocidade discretizada atual, dado a posição e a velocidade contínua
        # para não estourar o array, precisa usar o min e o max
        pos_discretizada = max(0, min(self.n_posicoes - 1, int((pos - POS_MINIMA) / self.tamanho_degrau_pos)))
        vel_discretizada = max(0, min(self.n_velocidades - 1, int((vel - VEL_MINIMA) / self.tamanho_degrau_vel)))
        return pos_discretizada, vel_discretizada

## Tarefa B: Implementação dos Agentes Tabulares
Implementem os algoritmos solicitados.

In [10]:
# ============================================================================
# TAREFA B: AGENTE SARSA(λ) COM TRAÇOS DE ELEGIBILIDADE  (Integrante 2)
# Requer: StateDiscretizer (Integrante 1) já definido numa célula anterior.
# ============================================================================
import numpy as np
import gymnasium as gym


def escolher_acao(Q, estado, epsilon, rng, n_acoes=3):
    """Política epsilon-greedy com desempate aleatório."""
    if rng.random() < epsilon:
        return int(rng.integers(n_acoes))
    q = Q[estado]  # estado é uma tupla (pos, vel) -> vetor com 3 valores
    return int(rng.choice(np.flatnonzero(q == q.max())))


def treinar_sarsa_lambda(
    discretizer,
    n_episodios=500,
    alpha=0.1,
    gamma=1.0,
    lam=0.9,
    epsilon_inicial=1.0,
    epsilon_minimo=0.01,
    epsilon_decay=0.99,
    tipo_traco="replacing",
    limiar_traco=1e-3,
    max_passos=200,
    seed=0,
):


    env = gym.make("MountainCar-v0", max_episode_steps=max_passos)
    rng = np.random.default_rng(seed)
    n_acoes = env.action_space.n
    Q = np.zeros((discretizer.n_posicoes, discretizer.n_velocidades, n_acoes))

    recompensa, passos, passos_acum = [], [], []
    q_updates, q_updates_acum = [], []
    total_passos, total_updates = 0, 0
    epsilon = epsilon_inicial

    for ep in range(n_episodios):
        obs, _ = env.reset(seed=int(rng.integers(1_000_000)))
        s = discretizer.discretize(obs)
        a = escolher_acao(Q, s, epsilon, rng, n_acoes)
        E = np.zeros_like(Q)  # (traços zerados a cada episódio)
        G, n_passos, n_updates = 0.0, 0, 0
        fim = False

        while not fim:
            obs2, r, terminou, truncou, _ = env.step(a)
            fim = terminou or truncou
            s2 = discretizer.discretize(obs2)
            a2 = escolher_acao(Q, s2, epsilon, rng, n_acoes)


            alvo = r if terminou else r + gamma * Q[s2 + (a2,)]
            delta = alvo - Q[s + (a,)]

            if tipo_traco == "replacing":
                E[s + (a,)] = 1.0
            else:
                E[s + (a,)] += 1.0

            ativos = E > 0
            Q[ativos] += alpha * delta * E[ativos]
            n_updates += int(ativos.sum())  # (custo computacional)

            E *= gamma * lam
            E[E < limiar_traco] = 0.0

            s, a = s2, a2
            G += r
            n_passos += 1

        total_passos += n_passos
        total_updates += n_updates
        recompensa.append(G)
        passos.append(n_passos)
        passos_acum.append(total_passos)
        q_updates.append(n_updates)
        q_updates_acum.append(total_updates)
        epsilon = max(epsilon_minimo, epsilon * epsilon_decay)

    env.close()
    metricas = {
        "recompensa": np.array(recompensa),
        "passos": np.array(passos),
        "passos_acum": np.array(passos_acum),
        "q_updates": np.array(q_updates),
        "q_updates_acum": np.array(q_updates_acum),
    }
    return Q, metricas


def rodar_seeds(discretizer, n_seeds=10, **kwargs):
    resultados = {}
    for seed in range(n_seeds):
        _, m = treinar_sarsa_lambda(discretizer, seed=seed, **kwargs)
        for k, v in m.items():
            resultados.setdefault(k, []).append(v)
    return {k: np.array(v) for k, v in resultados.items()}




In [33]:
def busca_hiperparametros(discretizer, grade, n_seeds=3, n_episodios=500):

    linhas = []

    for alpha, lam, decay in itertools.product(
        grade["alpha"],
        grade["lam"],
        grade["epsilon_decay"]
    ):

        r = rodar_seeds(
            discretizer,
            n_seeds=n_seeds,
            n_episodios=n_episodios,
            alpha=alpha,
            lam=lam,
            epsilon_decay=decay
        )

        linhas.append({
            "alpha": alpha,
            "lam": lam,
            "epsilon_decay": decay,
            "passos_medios": r["passos"].mean(),
            "passos_ultimos50": r["passos"][:, -50:].mean(),
            "q_updates_total": r["q_updates_acum"][:, -1].mean()
        })

    df = pd.DataFrame(linhas)

    df = df.sort_values(
        "passos_ultimos50"
    ).reset_index(drop=True)

    return df

In [34]:
import numpy as np


class StateDiscretizer:
    def __init__(self, n_posicoes, n_velocidades):
        self.n_posicoes = n_posicoes
        self.n_velocidades = n_velocidades

        # Limites do espaço de estados do MountainCar-v0
        self.posicao_min = -1.2
        self.posicao_max = 0.6

        self.velocidade_min = -0.07
        self.velocidade_max = 0.07

        # Cria os intervalos usados para discretizar cada dimensão
        self.posicoes = np.linspace(
            self.posicao_min,
            self.posicao_max,
            n_posicoes + 1
        )

        self.velocidades = np.linspace(
            self.velocidade_min,
            self.velocidade_max,
            n_velocidades + 1
        )

    def discretize(self, estado):
        posicao, velocidade = estado

        # Descobre em qual intervalo cada valor está
        pos = np.digitize(posicao, self.posicoes) - 1
        vel = np.digitize(velocidade, self.velocidades) - 1

        # Garante que os índices estejam dentro dos limites
        pos = np.clip(pos, 0, self.n_posicoes - 1)
        vel = np.clip(vel, 0, self.n_velocidades - 1)

        return (int(pos), int(vel))

In [35]:
disc_grossa = StateDiscretizer(10, 10)
disc_fina = StateDiscretizer(30, 30)

In [ ]:
estado = np.array([-0.5, 0.02])

print("Estado original:", estado)
print("Estado discretizado:", disc_grossa.discretize(estado))

In [ ]:
Q, m = treinar_sarsa_lambda(
    disc_grossa,
    n_episodios=300,
    seed=0
)

print(
    "Média de passos nos últimos 20 episódios:",
    m["passos"][-20:].mean()
)

In [ ]:
Q_fina, m_fina = treinar_sarsa_lambda(
    disc_fina,
    n_episodios=300,
    seed=0
)

print(
    "Média de passos nos últimos 20 episódios:",
    m_fina["passos"][-20:].mean()
)

In [28]:
import itertools
import numpy as np
import pandas as pd


In [42]:
grade = {
    "alpha": [0.05, 0.1, 0.2],
    "lam": [0.7, 0.9],
    "epsilon_decay": [0.99, 0.995],
    "epsilon_inicial": [0.1, 1.0],
    "tipo_traco": ["replacing", "accumulating"],
}

In [ ]:
teste = busca_hiperparametros(
    disc_grossa,
    {
        "alpha": [0.1],
        "lam": [0.9],
        "epsilon_decay": [0.99],
        "epsilon_inicial": [1.0],
        "tipo_traco": ["replacing"],
    },
    n_seeds=1,
    n_episodios=10,
)
print(teste)

In [ ]:
resultados_grossa = busca_hiperparametros(
    disc_grossa,
    grade,
    n_seeds=3,
    n_episodios=500
)

print(resultados_grossa.head(10))

In [ ]:
print(resultados_grossa.head(10))

In [ ]:
resultados_fina = busca_hiperparametros(
    disc_fina,
    grade,
    n_seeds=3,
    n_episodios=500
)

print(resultados_fina.head(10))

In [ ]:
melhor_grossa = resultados_grossa.iloc[0]
melhor_fina = resultados_fina.iloc[0]

print("Melhor configuração - Grossa:\n", melhor_grossa)
print("\nMelhor configuração - Fina:\n", melhor_fina)

# Execução final: as 10 seeds, uma vez, com os melhores parâmetros
def rodar_final(disc, melhor):
    return rodar_seeds(
        disc,
        n_seeds=10,
        seed_base=0,
        n_episodios=500,
        alpha=melhor["alpha"],
        lam=melhor["lam"],
        epsilon_decay=melhor["epsilon_decay"],
        epsilon_inicial=melhor["epsilon_inicial"],
        tipo_traco=melhor["tipo_traco"],
    )

res_grossa = rodar_final(disc_grossa, melhor_grossa)
res_fina = rodar_final(disc_fina, melhor_fina)

## 4.1 Gráficos de Curva de Aprendizado

Com os dados coletados, gerem os dois conjuntos de gráficos exigidos. Utilizem o `seaborn.lineplot`, pois ele calcula e desenha o Intervalo de Confiança de 95% automaticamente ao receber dados de múltiplas repetições (seeds).

In [ ]:
# ==============================================================================
# GRÁFICOS
# ==============================================================================


## 4.2 Análise de Resultados e Aliasing de Estados

*(Respondam de forma dissertativa, embasando-se nos gráficos gerados acima)*

**a) Qual o melhor algoritmo em termos de eficiência amostral?**
> Escreva sua resposta aqui...

**b) Qual o melhor algoritmo em termos de eficiência computacional?**
> Escreva sua resposta aqui...

**c) Como a modelagem de estados afetou especificamente o modelo construído pelo Dyna-Q em comparação com o SARSA(λ)? Pense na relação entre as transições reais do ambiente vs nos estados discretizados.**
> Escreva sua resposta detalhada sobre a modelagem de estados aqui...

In [ ]:
# ==============================================================================
# GRAVAÇÃO E EXIBIÇÃO DE VÍDEO DOS AGENTES TREINADOS
# ==============================================================================
# Salve um ou mais episódios dos seus agentes que valem a pena serem vistos

## 4.3 Diário de Bordo e Registro de Falhas


**a) O que notadamente falhou nas escolhas de hiperparâmetros (alpha, decaimento de epsilon, λ, N)? Por que falharam?**
> Escreva sua resposta aqui...

**b) Houve alguma dificuldade ou erros na implementação de alguma etapa do projeto? Como foi resolvido?**
> Escreva sua resposta aqui...

**c) O que mais vale a pena relatar que não foi coberto nas demais perguntas?**
> Escreva sua resposta aqui...